# 실습 2 — Vector Search 2.0 상품 검색 엔진 직접 돌려보기

**"배포할 에이전트의 검색 엔진을, 배포하기 전에 직접 돌려본다."**

이 노트북에서 실행하는 API 호출은 방금 Cloud Run에 배포를 시작한 쇼핑 에이전트(`app/embedding_vector.py`)가
런타임에 수행하는 것과 **동일한 호출**입니다. 마지막 두 셀에서 앱 소스를 직접 열어 한 줄씩 대조합니다.

| 항목 | 값 |
| :--- | :--- |
| 컬렉션 | `amazon-product-768-compact` (`asia-northeast1`) |
| 임베딩 모델 | `gemini-embedding-2`, 768차원 |
| Dense 벡터 필드 | `text_embedding` (상품 설명문) / `image_embedding` (상품 이미지) |
| ANN 인덱스 | `idx-text-embedding`, `idx-image-embedding` (ScaNN) |

> [!IMPORTANT]
> - 이 노트북은 **컬렉션이나 인덱스를 만들지 않습니다.** 실습 시작 전에 실행한 `install.sh`가 이미 전부 만들어 두었습니다.
> - 시작 전에 `part2/README.md`의 **Cloud Run 배포 명령을 먼저 실행**해 두세요. 빌드(약 5분)가 이 실습과 병렬로 진행됩니다.


## 1. 클라이언트 초기화 및 컬렉션 핸들

`app/common.py` / `app/embedding_vector.py`가 모듈 로드 시점에 만드는 것과 **같은 클라이언트 4종**을 만듭니다.

| 클라이언트 | 역할 |
| :--- | :--- |
| `genai.Client` | Gemini Embedding 2로 **질의 벡터** 생성 (텍스트·이미지 공용) |
| `DataObjectSearchServiceClient` | 벡터 검색 / 배치 검색(RRF) |
| `DataObjectServiceClient` | 개별 상품 조회 |
| `RankServiceClient` | Vertex AI Ranking API 리랭킹 |

In [ ]:
import io
import statistics
import urllib.request
from html import escape
from pathlib import Path
from time import perf_counter

import google.auth
from google import genai
from google.genai import types
from google.cloud import vectorsearch_v1beta as vectorsearch
from google.cloud import discoveryengine_v1 as discoveryengine
from IPython.display import HTML, display
from PIL import Image

_, PROJECT_ID = google.auth.default()

# ── app/common.py 와 동일한 값 ──────────────────────────────────────────
LOCATION = "asia-northeast1"
COLLECTION_ID = "amazon-product-768-compact"
COLLECTION_NAME = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}"
IMAGE_SERVER = "https://thumbnail.aidemo.dev"

# ── app/embedding_vector.py 와 동일한 값 ────────────────────────────────
EMBEDDING_MODEL = "gemini-embedding-2"
OUTPUT_DIMENSIONALITY = 768
TEXT_FIELD = "text_embedding"
IMAGE_FIELD = "image_embedding"
RANKING_CONFIG = f"projects/{PROJECT_ID}/locations/global/rankingConfigs/default_ranking_config"
TEXT_QUERY_HYBRID_WEIGHTS = [1.35, 0.65]
IMAGE_QUERY_HYBRID_WEIGHTS = [0.65, 1.35]

embedding_client = genai.Client(vertexai=True, project=PROJECT_ID, location="global")
search_client = vectorsearch.DataObjectSearchServiceClient()
data_client = vectorsearch.DataObjectServiceClient()
rank_client = discoveryengine.RankServiceClient()

print("PROJECT_ID :", PROJECT_ID)
print("COLLECTION :", COLLECTION_NAME)

## 2. 백그라운드 인덱싱 완료 확인

실습 맨 처음에 실행한 `install.sh`는 `session2_index_builder.py`를 **백그라운드(nohup)** 로 띄워 놓았습니다.
그 스크립트가 수행한 4단계는 다음과 같습니다.

1. Collection 생성 (`data_schema` = `name`/`description`, `vector_schema` = 768차원 dense 2개)
2. GCS → `ImportDataObjects` (상품 레코드 + 사전 계산된 임베딩 2종)
3. `idx-text-embedding` ScaNN 인덱스 생성
4. `idx-image-embedding` ScaNN 인덱스 생성

각 단계는 LRO(Long Running Operation)로 실행되므로, 아래 명령으로 **완료 여부(`done`)** 를 확인할 수 있습니다.

In [ ]:
!gcloud vector-search operations list --location=asia-northeast1

In [ ]:
# 컬렉션에 실제로 상품이 몇 건 적재되었는지 확인합니다. (벡터 없이 집계만 수행)
try:
    response = search_client.aggregate_data_objects(
        vectorsearch.AggregateDataObjectsRequest(parent=COLLECTION_NAME, aggregate="COUNT")
    )
    # aggregate_results 는 Struct 리스트로 돌아옵니다. dict 로 풀어야 읽을 수 있습니다.
    rows = [dict(row) for row in response.aggregate_results]
    print("적재된 상품 수 :", rows)
except Exception as exc:  # 임포트가 아직 진행 중이면 여기로 들어옵니다.
    print("❌ 집계 실패:", exc)
    print()
    print("확인 순서:")
    print("  1) 위 4번 셀의 operations 목록에서 임포트 작업이 done: true 인지")
    print("  2) 터미널에서  tail -30 ~/smx-multimodal-agent/index_builder.log")
    print("  3) 그래도 비어 있으면  bash install.sh  를 다시 실행")


## 3. 컬렉션 스키마 확인 — dense 벡터 필드가 **2개**인 이유

Part 1의 미디어 컬렉션은 dense 필드가 1개였습니다. 이 상품 컬렉션은 **2개**입니다.

| 필드 | 무엇을 임베딩했나 | 무엇에 강한가 |
| :--- | :--- | :--- |
| `text_embedding` | 상품명 + 설명문 | "방수 등산화" 처럼 **속성·용도 언어**로 찾을 때 |
| `image_embedding` | 상품 대표 이미지 | "이거랑 똑같이 생긴 것" 처럼 **생김새**로 찾을 때 |

Gemini Embedding 2는 텍스트와 이미지를 **같은 벡터 공간**에 임베딩하므로,
하나의 질의 벡터를 **두 필드 모두에** 던질 수 있습니다. 7번 셀의 하이브리드 검색이 정확히 그것을 합니다.

In [ ]:
try:
    service_client = vectorsearch.VectorSearchServiceClient()
    collection = service_client.get_collection(name=COLLECTION_NAME)
    print("── data_schema (검색 결과로 돌려받을 수 있는 데이터 필드) ──")
    print(collection.data_schema)
    print("── vector_schema (검색 대상 벡터 필드) ──")
    print(collection.vector_schema)
except Exception as exc:
    print("컬렉션 조회 실패:", exc)

## 4. 상품 카탈로그 프리뷰 + 공용 헬퍼 정의

앞으로 계속 쓸 함수 3개를 정의합니다. 셋 다 `app/embedding_vector.py`의 함수와 **1:1로 대응**합니다.

| 노트북 함수 | 앱 함수 |
| :--- | :--- |
| `embed()` | `_embed_with_gemini_embedding_2()` |
| `vector_search()` | `_text_similarity_collection_search()` / `_image_similarity_collection_search()` |
| `to_item()` | `_search_result_to_dict()` |

> 이 컬렉션에는 서버측 자동 임베딩(`vertex_embedding_config`)이 설정되어 있지 않습니다.
> 따라서 `search_text=...`를 넘기는 `SemanticSearch`가 아니라, **클라이언트에서 벡터를 만들어 넣는
> `VectorSearch`(bring-your-own-vector)** 를 사용합니다. 에이전트도 정확히 이 방식으로 동작합니다.

In [ ]:
def embed(text: str | None = None, image: bytes | None = None) -> list[float]:
    """app/embedding_vector.py 의 _embed_with_gemini_embedding_2() 와 동일."""
    contents = text if text is not None else types.Part.from_bytes(data=image, mime_type="image/jpeg")
    response = embedding_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=contents,
        config=types.EmbedContentConfig(output_dimensionality=OUTPUT_DIMENSIONALITY),
    )
    return list(response.embeddings[0].values)


def to_item(result) -> dict:
    """app/embedding_vector.py 의 _search_result_to_dict() 와 동일."""
    obj = result.data_object
    item_id = obj.data_object_id or obj.name.split("/")[-1]
    return {
        "id": item_id,
        "name": str(obj.data.get("name", "")),
        "description": str(obj.data.get("description", "")),
        "score": result.distance,
    }


def vector_search(embedding, search_field, top_k=8, metadata_filter=None) -> list[dict]:
    """app/embedding_vector.py 의 _text/_image_similarity_collection_search() 와 동일."""
    clause_kwargs = {
        "search_field": search_field,
        "vector": vectorsearch.DenseVector(values=embedding),
        "top_k": top_k,
        "output_fields": vectorsearch.OutputFields(data_fields=["name", "description"]),
    }
    if metadata_filter is not None:  # 9번 셀에서 사용합니다.
        clause_kwargs["filter"] = metadata_filter
    request = vectorsearch.SearchDataObjectsRequest(
        parent=COLLECTION_NAME,
        vector_search=vectorsearch.VectorSearch(**clause_kwargs),
    )
    response = search_client.search_data_objects(request)
    return [to_item(result) for result in response.results]


def dedupe(items):
    """상품명이 같은 항목을 하나만 남깁니다.

    Amazon 카탈로그에는 색상·사이즈 변형이 서로 다른 ID로 들어 있어서, 그대로 그리면
    상위 10건 중 8건이 같은 이름으로 보입니다. 순위가 바뀌어도 화면이 안 바뀐 것처럼
    오해되므로 표시 단계에서만 걸러냅니다. (검색 자체는 원본 결과를 그대로 씁니다.)
    """
    seen, unique = set(), []
    for item in items:
        key = item["name"].strip().lower()
        if key not in seen:
            seen.add(key)
            unique.append(item)
    return unique


def render(items, title="", limit=8):
    """검색 결과를 썸네일 그리드로 표시합니다."""
    items = dedupe(items)
    cards = []
    for rank, item in enumerate(items[:limit], 1):
        cards.append(
            "<div style='width:148px;margin:6px;font-size:11px;text-align:center'>"
            "<img src='{}/{}.webp' style='width:140px;height:140px;object-fit:contain;"
            "background:#fff;border:1px solid #eee'>"
            "<div><b>{}.</b> {}</div><div style='color:#888'>{:.4f}</div></div>".format(
                IMAGE_SERVER, item["id"], rank, escape(item["name"])[:64], item["score"]
            )
        )
    display(HTML(
        "<b>{}</b><div style='display:flex;flex-wrap:wrap'>{}</div>".format(
            escape(title), "".join(cards))))


preview = vector_search(embed(text="summer floral dress"), TEXT_FIELD, top_k=8)
render(preview, "카탈로그 프리뷰 — 'summer floral dress'")

## 5. 텍스트 질의 ➔ `text_embedding` 필드 검색

에이전트가 `find_items` 툴로 넘겨받은 **영어 텍스트 쿼리**를 처리하는 경로입니다.

키워드 매칭이 아니라 의미 매칭이라는 점을 확인하세요. 두 번째 질의에는 `thermos`, `mug` 같은 단어가
한 글자도 들어 있지 않지만, 보온 용기류가 상위로 올라옵니다.

In [ ]:
QUERIES = [
    "waterproof hiking shoes for rainy trails",
    "something that keeps my coffee hot on the desk all morning",
]

text_results = []
for query in QUERIES:
    embed_started = perf_counter()
    query_vector = embed(text=query)
    embed_ms = (perf_counter() - embed_started) * 1000

    search_started = perf_counter()
    results = vector_search(query_vector, TEXT_FIELD, top_k=8)
    search_ms = (perf_counter() - search_started) * 1000

    print("query={!r}  embed_ms={:.1f}  search_ms={:.1f}  results={}".format(
        query, embed_ms, search_ms, len(results)))
    render(results, "text_embedding ← " + query)
    if not text_results:
        text_results = results

## 6. 이미지 질의 ➔ `image_embedding` 필드 검색 (크로스모달)

에이전트는 **스마트폰 카메라 프레임(JPEG)** 을 그대로 임베딩해서 검색합니다
(`app/main.py`의 유사상품 워커 ➔ `_image_similarity_search()`).
여기서는 카탈로그 썸네일 한 장을 카메라 프레임 대신 사용합니다.

핵심은 **같은 이미지 벡터 하나를 두 필드에 각각 던져 본다**는 점입니다.

- `image_embedding` 검색 ➔ 생김새가 닮은 상품 (질의로 쓴 상품 자신이 1등으로 나오는 게 정상입니다)
- `text_embedding` 검색 ➔ **이미지 벡터로 설명문 벡터를 찾는 크로스모달 검색**

두 결과가 서로 다르다는 것이, 다음 셀의 RRF 융합이 필요한 이유입니다.

In [ ]:
def fetch_jpeg(url: str) -> bytes:
    """썸네일을 내려받아 JPEG 바이트로 변환합니다 (앱이 카메라에서 받는 형식과 동일)."""
    with urllib.request.urlopen(url) as response:
        raw = response.read()
    buffer = io.BytesIO()
    Image.open(io.BytesIO(raw)).convert("RGB").save(buffer, format="JPEG")
    return buffer.getvalue()


seed = text_results[0]
seed_url = "{}/{}.webp".format(IMAGE_SERVER, seed["id"])
print("질의 이미지 :", seed["name"])
display(HTML("<img src='{}' width='180' style='border:1px solid #eee'>".format(seed_url)))

image_vector = embed(image=fetch_jpeg(seed_url))
print("이미지 질의 벡터 차원 :", len(image_vector))

render(vector_search(image_vector, IMAGE_FIELD, top_k=8), "① image_embedding ← 이미지 벡터 (생김새)")
render(vector_search(image_vector, TEXT_FIELD, top_k=8), "② text_embedding ← 이미지 벡터 (크로스모달)")

## 7. [핵심] 하이브리드 검색과 RRF 가중치 실험

`batch_search_data_objects`는 여러 검색 절(clause)을 **한 번의 왕복**으로 실행하고,
서버 내장 **RRF(Reciprocal Rank Fusion)** 로 순위를 융합합니다.

$$\text{score}(d) = \sum_{i} w_i \cdot \frac{1}{k + \text{rank}_i(d)}$$

Part 1에서 손으로 조정하던 `alpha`가, 여기서는 `ReciprocalRankFusion(weights=[...])` 입니다.
**절의 순서가 곧 가중치의 순서**입니다. 아래 함수는 `_hybrid_collection_search()`와 절 순서까지 동일합니다.

| | 1번 절 (`text_embedding`) | 2번 절 (`image_embedding`) |
| :--- | :--- | :--- |
| `TEXT_QUERY_HYBRID_WEIGHTS` | **1.35** | 0.65 |
| `IMAGE_QUERY_HYBRID_WEIGHTS` | 0.65 | **1.35** |

앱은 **질의가 텍스트면 설명문 쪽에, 이미지면 생김새 쪽에 무게를 싣습니다.**
같은 질의 벡터로 가중치만 뒤집어 보고, 순위가 어떻게 흔들리는지 직접 확인하세요.

In [ ]:
def hybrid_search(embedding, weights, top_k=20) -> list[dict]:
    """app/embedding_vector.py 의 _hybrid_collection_search() 와 동일한 요청."""
    request = vectorsearch.BatchSearchDataObjectsRequest(
        parent=COLLECTION_NAME,
        searches=[
            vectorsearch.Search(  # 1번 절 → weights[0]
                vector_search=vectorsearch.VectorSearch(
                    search_field=TEXT_FIELD,
                    vector=vectorsearch.DenseVector(values=embedding),
                    top_k=top_k,
                    output_fields=vectorsearch.OutputFields(data_fields=["name", "description"]),
                )
            ),
            vectorsearch.Search(  # 2번 절 → weights[1]
                vector_search=vectorsearch.VectorSearch(
                    search_field=IMAGE_FIELD,
                    vector=vectorsearch.DenseVector(values=embedding),
                    top_k=top_k,
                    output_fields=vectorsearch.OutputFields(data_fields=["name", "description"]),
                )
            ),
        ],
        combine=vectorsearch.BatchSearchDataObjectsRequest.CombineResultsOptions(
            ranker=vectorsearch.Ranker(
                rrf=vectorsearch.ReciprocalRankFusion(weights=weights)
            ),
            output_fields=vectorsearch.OutputFields(data_fields=["name", "description"]),
            top_k=top_k,
        ),
    )
    response = search_client.batch_search_data_objects(request)
    fused = response.results[0].results if response.results else []
    return [to_item(result) for result in fused]


def compare(left_title, left, right_title, right, limit=8):
    """두 결과 리스트를 나란히 놓고 순위 변동을 표시합니다."""
    left, right = dedupe(left), dedupe(right)
    left_rank = {item["id"]: i for i, item in enumerate(left, 1)}
    right_rank = {item["id"]: i for i, item in enumerate(right, 1)}

    def badge(item, rank, other):
        previous = other.get(item["id"])
        if previous is None:
            return "<span style='color:#c0392b'>NEW</span>"
        if previous == rank:
            return "<span style='color:#aaa'>=</span>"
        if previous > rank:
            return "<span style='color:#1e8449'>▲{}</span>".format(previous - rank)
        return "<span style='color:#2471a3'>▼{}</span>".format(rank - previous)

    def cells(items, rank, other):
        if rank > len(items):
            return "<td></td><td></td>"
        item = items[rank - 1]
        return ("<td style='padding:4px'><img src='{}/{}.webp' width='52' "
                "style='object-fit:contain;background:#fff'></td>"
                "<td style='padding:4px;font-size:11px'>{} {}</td>").format(
                    IMAGE_SERVER, item["id"], escape(item["name"])[:46], badge(item, rank, other))

    rows = []
    for rank in range(1, limit + 1):
        rows.append("<tr><td style='padding:4px;color:#888'>{}</td>{}{}</tr>".format(
            rank, cells(left, rank, right_rank), cells(right, rank, left_rank)))
    display(HTML(
        "<table style='border-collapse:collapse'>"
        "<tr><th></th><th colspan='2' style='padding:6px'>{}</th>"
        "<th colspan='2' style='padding:6px'>{}</th></tr>{}</table>".format(
            escape(left_title), escape(right_title), "".join(rows))))


text_weighted = hybrid_search(image_vector, TEXT_QUERY_HYBRID_WEIGHTS)
image_weighted = hybrid_search(image_vector, IMAGE_QUERY_HYBRID_WEIGHTS)

compare(
    "TEXT_QUERY_HYBRID_WEIGHTS = [1.35, 0.65]", text_weighted,
    "IMAGE_QUERY_HYBRID_WEIGHTS = [0.65, 1.35]", image_weighted,
)
print("▲▼ 는 반대편 목록 대비 순위 변동, NEW 는 반대편 상위 8위 안에 없던 상품입니다.")

> **직접 해보기**: `hybrid_search(image_vector, [2.0, 0.0])` 과 `hybrid_search(image_vector, [0.0, 2.0])` 을
> 비교해 보세요. 가중치를 0으로 주면 해당 절이 사실상 무력화되어, 단일 필드 검색과 같아집니다.
> 어느 쪽 극단도 정답이 아니라는 점이 하이브리드를 쓰는 이유입니다.

## 8. Ranking API 리랭킹

벡터 검색은 **재현율(recall)** 을 담당하고, Ranking API는 **정밀도(precision)** 를 담당합니다.
`find_items` 툴이 `ranking_query`(짧은 영어 요약)를 따로 받는 이유가 이것입니다.

- 벡터 검색: 100건을 빠르게 긁어온다 (근사)
- Ranking API: 그 100건을 질의와 **교차 인코딩**으로 정독해 다시 세운다 (정확)

아래 `rank_results()`는 `app/embedding_vector.py`의 `_rank_results()`와 동일합니다.

In [ ]:
def rank_results(query: str, results: list[dict]) -> list[dict]:
    """app/embedding_vector.py 의 _rank_results() 와 동일 (원본은 리스트를 제자리 정렬)."""
    if not results or not query:
        return results
    records = [
        discoveryengine.RankingRecord(
            id=item["id"], title=item["name"], content=item.get("description", "")
        )
        for item in results
    ]
    response = rank_client.rank(
        request=discoveryengine.RankRequest(
            ranking_config=RANKING_CONFIG,
            query=query,
            records=records,
            top_n=len(records),
        )
    )
    scores = {record.id: record.score for record in response.records}
    ranked = [dict(item, score=scores.get(item["id"], 0.0)) for item in results]
    ranked.sort(key=lambda item: item["score"], reverse=True)
    return ranked


# 13번 셀의 질의 이미지는 QUERIES[0] 검색 결과에서 골랐습니다.
# 리랭킹 질의도 같은 의도여야 순위 변동이 의미를 갖습니다.
RANKING_QUERY = QUERIES[0]

reranked = rank_results(RANKING_QUERY, text_weighted)
compare("RRF 융합 직후", text_weighted, "Ranking API 리랭킹 후 — " + RANKING_QUERY, reranked)
print("점수 스케일도 바뀝니다: RRF 융합 점수 → Ranking API 관련도 점수(0~1).")

## 9. 메타데이터 필터 결합

VS2 필터는 **MongoDB 스타일 JSON**이며, 검색 절(clause)마다 개별로 지정합니다.
(`$eq`, `$ne`, `$lt`, `$gt`, `$in`, `$and`, `$or` …)

```python
filter={"$and": [{"category": {"$eq": "Shorts"}}, {"retail_price": {"$lt": 30}}]}
```

> [!NOTE]
> 이 실습 컬렉션의 `data_schema`에는 `name`과 `description` **두 필드밖에 없습니다**
> (`session2_index_builder.py` Step 1 참고). 그래서 아래 예제는 `name`으로 필터링합니다.
> 실제 서비스라면 `category`, `retail_price`, `brand` 같은 속성을 `data_schema`에 추가하고,
> 인덱스 생성 시 `Index(index_field=..., filter_fields=["category", "retail_price"])` 로
> **필터 대상 필드를 선언**해 두어야 대규모에서도 빠르게 걸러집니다.

In [ ]:
allowed_names = [item["name"] for item in preview[:3]]
print("필터로 허용할 상품 3건:")
for name in allowed_names:
    print("  -", name[:70])

filtered = vector_search(
    embed(text="summer floral dress"),
    TEXT_FIELD,
    top_k=8,
    metadata_filter={"name": {"$in": allowed_names}},
)
print("\n필터 적용 후 결과 수:", len(filtered), "(유사도와 무관하게 허용 목록 밖 상품은 제외됩니다)")
render(filtered, "text_embedding + filter={'name': {'$in': [...]}}")

## 10. ANN(ScaNN) vs kNN — **코드는 그대로, 속도만 바뀐다**

| | Part 1 미디어 컬렉션 | Part 2 상품 컬렉션 |
| :--- | :--- | :--- |
| 인덱스 | 없음 | ScaNN 2개 (`idx-text-embedding`, `idx-image-embedding`) |
| 검색 방식 | **kNN 완전탐색** — 모든 벡터와 비교 | **ANN 근사탐색** — 후보군만 비교 |
| 데이터 규모 | 약 1,000건 | 수만 건 |
| 질의 코드 | `SearchDataObjectsRequest(...)` | **완전히 동일** |

Vector Search 2.0에서 인덱스는 **별도의 검색 엔드포인트가 아닙니다.** 검색은 언제나 컬렉션을 향하고,
질의 필드에 인덱스가 있으면 서버가 알아서 사용합니다. 그래서 Part 1 코드를 한 줄도 고치지 않고
프로덕션 규모로 넘어갈 수 있습니다 — *transparent upgrade*.

In [ ]:
def find_file(*candidates) -> Path:
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    raise FileNotFoundError(candidates)


def show_function(path: Path, name: str) -> None:
    lines = path.read_text().splitlines()
    start = next(i for i, line in enumerate(lines) if line.startswith("def " + name + "("))
    end = start + 1
    while end < len(lines):
        line = lines[end]
        if line.strip() and not line[:1].isspace():   # 들여쓰기가 끝나면 함수도 끝
            break
        end += 1
    print("─" * 78)
    print("{}  ::  {}()".format(path, name))
    print("─" * 78)
    print("\n".join(lines[start:end]).rstrip())


BUILDER_PY = find_file("../session2_index_builder.py", "session2_index_builder.py")
show_function(BUILDER_PY, "request_index")

# 이 컬렉션에 실제로 어떤 인덱스가 붙어 있는지 먼저 확인합니다.
# 인덱스가 없는 필드로 검색하면 kNN 완전탐색으로 처리되므로 아래 지연시간의 의미가 달라집니다.
indexes = list(service_client.list_indexes(parent=COLLECTION_NAME))
if indexes:
    for index in indexes:
        print("인덱스 {}  ← index_field={}  store_fields={}".format(
            index.name.split("/")[-1], index.index_field, list(index.store_fields)))
else:
    print("⚠️ 인덱스가 아직 없습니다. 아래 지연시간은 ANN이 아니라 kNN 완전탐색 수치입니다.")
print()

# 질의의 실제 지연시간을 측정합니다.
probe_vector = embed(text="wireless noise cancelling headphones")
latencies = []
for _ in range(5):
    started = perf_counter()
    vector_search(probe_vector, TEXT_FIELD, top_k=20)
    latencies.append((perf_counter() - started) * 1000)

print("\nsearch_ms 5회 :", ", ".join("{:.1f}".format(value) for value in latencies))
print("중앙값        : {:.1f} ms  (임베딩 생성 시간 제외, 순수 검색)".format(statistics.median(latencies)))

## 11. `app/embedding_vector.py` 소스 대조 — 그리고 실제로 내려진 결정

7번 셀에서 정의한 `hybrid_search()`와, 배포 중인 앱의 `_hybrid_collection_search()`를 나란히 놓습니다.
`parent`, 절 순서(`text_embedding` ➔ `image_embedding`), `output_fields`, `weights`, 결과 파싱까지
**같은 요청**입니다. 다른 점은 `top_k`(앱은 `SEARCH_TOP_K = 100`)와 로깅뿐입니다.

그런데 이 함수는 **런타임에 호출되지 않습니다.** 왜 그런지가 이 실습의 마지막 이야기입니다.


In [ ]:
import inspect

APP_EMBEDDING_PY = find_file(
    "app/embedding_vector.py",
    "../part2/app/embedding_vector.py",
)

show_function(APP_EMBEDDING_PY, "_hybrid_collection_search")
print()
print("─" * 78)
print("이 노트북의 hybrid_search()")
print("─" * 78)
print(inspect.getsource(hybrid_search).rstrip())

> [!IMPORTANT]
> ### 재현율 vs 지연 — 이 앱에서 실제로 내려진 결정
>
> 방금 대조한 `_hybrid_collection_search()`는 구현되어 있지만 **실행되지 않습니다.**
> `_collection_search()`가 `_text_similarity_collection_search()`로 단축되어 있고,
> 하이브리드 호출부는 주석 처리되어 있습니다.
>
> **실수가 아닙니다.** 원저자가 커밋 `4bc2657 — "Set use only text for latency reduce"` 로
> 의도적으로 바꾼 것입니다. 무엇을 주고 무엇을 받았는지 보세요.
>
> | | 하이브리드 (원래) | 텍스트 단독 (현재) |
> | :--- | :--- | :--- |
> | 질의 임베딩 | 1회 | 1회 (동일) |
> | 서버측 벡터 검색 | **2회** (`text_embedding` + `image_embedding`) | 1회 |
> | RRF 융합 | 있음 | 없음 |
>
> - **잃은 것 — 크로스모달 재현율.** 7번 셀에서 두 필드의 상위 결과가 거의 겹치지 않는 것을 보셨죠.
>   단독 경로는 `image_embedding` 쪽 후보를 **통째로** 보지 못합니다.
>   *"설명문에는 안 적혀 있지만 생김새가 딱 맞는 상품"* 이 결과에서 사라집니다.
> - **얻은 것 — 응답 지연.** 매 질의마다 수만 건 규모의 벡터 검색이 하나 줄어듭니다.
>   Gemini Live 음성 대화에서 수백 ms는 대화 흐름이 끊기느냐 마느냐의 차이입니다.
>   게다가 `find_items`는 쿼리를 여러 개 **동시에** 던지므로 부하가 그만큼 배로 붙습니다.
> - **영향 없는 것 — 카메라 경로.** `_image_similarity_search()`는 원래부터 `image_embedding`
>   단독이라 이 결정과 무관하게 그대로 동작합니다.
>
> 벡터 검색 설계는 좋은 기능을 전부 켜는 일이 아니라,
> **어느 재현율을 어느 지연에 팔지 고르는 일**입니다. 정답은 서비스마다 다릅니다.
>
> 아래 셀에서 단축된 실제 코드를 확인하세요. 주석을 되살리면 하이브리드가 그대로 켜집니다.
> 실제로 얼마나 느려지는지 궁금하다면, 배포된 앱의 `/test/find_items` 엔드포인트로 직접 재볼 수 있습니다.


In [ ]:
show_function(APP_EMBEDDING_PY, "_collection_search")

## 12. 에이전트 프롬프트와 `find_items` 툴 호출 흐름

마지막으로 검색 엔진이 **에이전트 안에서 어떻게 불리는지** 정리합니다.
LensMosaic에는 검색 경로가 두 개 있고, 둘 다 방금 실습한 함수를 사용합니다.

```
[경로 A] 카메라 프레임 (항상, 자동)
  브라우저 JPEG 프레임
    → app/main.py 유사상품 워커 (SIMILAR_SEARCH_WORKER_COUNT 개 스레드)
    → _image_similarity_search(image)
    → _image_similarity_collection_search()   # image_embedding 단독  ← 6번 셀 ①
    → 화면 좌측 타일 실시간 갱신

[경로 B] 음성 발화 → Gemini Live 툴 호출
  "어울리는 가방 추천해줘"
    → (필요 시) google_search 로 트렌드 수집
    → find_items(queries=[영어 쿼리 여러 개], ranking_query="짧은 영어 요약")
    → 쿼리마다 스레드 1개로 병렬 _collection_search(text=q, rerank=False)   ← 5번 셀
    → id 기준 중복 제거
    → _rank_results(ranking_query, items)                                   ← 8번 셀
    → 상위 MAX_TILE_ITEMS(64)건을 화면에 렌더링 + 음성 브리핑
```

프롬프트가 **영어 쿼리**를 만들도록 강제하는 이유는, 상품 설명문이 영어이기 때문입니다
(질의와 문서의 언어를 맞추면 임베딩 정합도가 올라갑니다).
반대로 사용자에게 들려주는 **음성 응답은 100% 한글**로 강제됩니다 — TTS 음성이 끊기는 것을 막기 위한 제약입니다.

In [ ]:
APP_PROMPT_PY = find_file(
    "app/prompt.py",
    "../part2/app/prompt.py",
)

prompt_source = APP_PROMPT_PY.read_text()
step1 = "## 1단계" + prompt_source.split("## 1단계", 1)[1].split("## 2단계", 1)[0]
print("─" * 78)
print("{}  ::  AGENT_PROMPT 발췌".format(APP_PROMPT_PY))
print("─" * 78)
print(step1.rstrip())

## 실습 2 완료 🎉

방금 확인한 것:

1. `install.sh`가 만들어 둔 컬렉션에 ScaNN 인덱스가 붙어 있고, 질의 코드는 Part 1과 동일하다.
2. **하나의 Gemini Embedding 2 벡터**를 `text_embedding` / `image_embedding` 두 필드에 모두 던질 수 있다.
3. `batch_search_data_objects` + **RRF weights** 가 Part 1에서 손으로 짠 `alpha` 를 대체한다.
4. 가중치를 뒤집으면 **순위가 실제로 바뀐다** — 튜닝 가능한 실제 손잡이다.
5. Ranking API가 재현율 위주 결과를 정밀도 위주로 다시 세운다.
6. 그리고 실제 서비스는 이 손잡이들을 **전부 켜지 않는다.** 이 앱은 실시간 음성 응답을 지키려고
   크로스모달 재현율을 내려놓았다 — 검색 설계는 기능 선택이 아니라 **트레이드오프 선택**이다.

이제 `part2/README.md`로 돌아가 Cloud Run 배포 상태를 확인하고, QR 코드를 만들어
스마트폰에서 에이전트를 직접 사용해 보세요.
